# Customer Churn Analytical Data Architecture

This notebook designs the analytical data architecture for the customer churn project. It loads the three raw datasets, inspects their columns and identifiers, defines the business purpose of each source, and proposes an industry-style relational schema for downstream analytics.

Model training is intentionally out of scope.

## 1. Load Source Datasets

Raw data should remain immutable. The first step is to load each CSV from `data/raw` and inspect shape, column names, types, nulls, and candidate identifiers.

In [ ]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path('../../data/raw')

telco = pd.read_csv(RAW_DIR / 'telco_customer_churn.csv')
activity = pd.read_csv(RAW_DIR / 'customer_activity.csv')
support = pd.read_csv(RAW_DIR / 'support_tickets.csv')

datasets = {
    'telco_customer_churn': telco,
    'customer_activity': activity,
    'support_tickets': support,
}

for name, df in datasets.items():
    print(f'{name}: {df.shape}')

In [ ]:
def inspect_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        'column': df.columns,
        'dtype': [df[col].dtype for col in df.columns],
        'null_count': [df[col].isna().sum() for col in df.columns],
        'unique_count': [df[col].nunique(dropna=True) for col in df.columns],
    })

for name, df in datasets.items():
    print(f'\n{name}')
    display(inspect_dataframe(df))
    display(df.head())

## 2. Candidate Customer Identifiers

A churn architecture should be explicit about customer identity. These datasets come from different operational contexts and do not share one confirmed customer key.

In [ ]:
identifier_checks = {
    'telco.customerID_unique': telco['customerID'].nunique(dropna=True),
    'telco.customerID_duplicates': telco['customerID'].duplicated().sum(),
    'activity.Customer ID_unique': activity['Customer ID'].nunique(dropna=True),
    'activity.Customer ID_nulls': activity['Customer ID'].isna().sum(),
    'activity.Invoice_unique': activity['Invoice'].nunique(dropna=True),
    'support.Ticket ID_unique': support['Ticket ID'].nunique(dropna=True),
    'support.Ticket ID_duplicates': support['Ticket ID'].duplicated().sum(),
    'support.Customer Email_unique': support['Customer Email'].nunique(dropna=True),
    'support.Customer Email_duplicates': support['Customer Email'].duplicated().sum(),
}

pd.Series(identifier_checks).to_frame('value')

Identifier findings:

- `telco_customer_churn.customerID` is the natural key for the telco source and is unique at customer snapshot grain.
- `customer_activity.Customer ID` is a source-specific customer identifier, but it contains nulls and repeats across invoice lines.
- `customer_activity.Invoice` is a transaction identifier, not a customer identifier.
- `support_tickets.Ticket ID` is the natural support ticket key.
- `support_tickets.Customer Email` is the best available support customer identifier, but it cannot safely join to telco `customerID` without an identity bridge.

## 3. Business Purpose by Dataset

### Telco Customer Churn

The core labeled churn dataset. It contains customer demographics, subscribed services, billing attributes, tenure, charges, and the churn outcome. Its grain is one row per telco customer subscription snapshot.

### Customer Activity

A transaction activity dataset at invoice-line grain. It supports behavior features such as recency, frequency, monetary value, product diversity, return behavior, and country-level activity.

### Support Tickets

A support interaction dataset at ticket grain. It supports experience features such as ticket count, unresolved ticket count, priority mix, response time, resolution time, channel preference, and satisfaction rating.

## 4. Relational Analytical Design

The design uses a dimensional model: facts store measurable events or snapshots, dimensions store descriptive context, and a bridge table resolves source-specific customer identifiers to canonical analytical customers.

### Dimension Tables

| Table | Primary Key | Granularity | Purpose |
|---|---|---|---|
| `dim_customer` | `customer_key` | One row per canonical customer | Master customer entity |
| `bridge_customer_identity` | `customer_identity_key` | One row per source customer identity | Maps source IDs/emails/names to `customer_key` |
| `dim_product` | `product_key` | One row per standardized product/SKU | Product context for activity and support |
| `dim_date` | `date_key` | One row per calendar date | Shared calendar dimension |
| `dim_contract` | `contract_key` | One row per contract and billing combination | Telco contract/payment context |
| `dim_service_plan` | `service_plan_key` | One row per telco service bundle | Telco services context |
| `dim_support_classification` | `support_classification_key` | One row per ticket classification combination | Ticket type, subject, priority, channel, status |

### Fact Tables

| Table | Primary Key | Foreign Keys | Granularity | Measures |
|---|---|---|---|---|
| `fact_customer_subscription_snapshot` | `subscription_snapshot_key` | `customer_key`, `contract_key`, `service_plan_key` | One row per telco customer snapshot | tenure, monthly charges, total charges, churn flag |
| `fact_customer_activity_line` | `activity_line_key` | `customer_key`, `product_key`, `invoice_date_key` | One row per invoice line item | quantity, unit price, line amount, return flag |
| `fact_support_ticket` | `ticket_key` | `customer_key`, `product_key`, `support_classification_key`, date keys | One row per support ticket | response minutes, resolution minutes, satisfaction, open/closed flags |

## 5. Final Analytical Table: `churn_features_master`

`churn_features_master` is the final customer-level feature table used by BI, analytics, and later modeling. It should be built from cleaned dimensions and aggregated facts, not directly from raw CSV joins.

Recommended grain: one row per `customer_key` per `feature_snapshot_date`.

Recommended primary key: (`customer_key`, `feature_snapshot_date`).

| Feature Group | Example Columns |
|---|---|
| Identity | `customer_key`, `feature_snapshot_date`, `source_systems_present` |
| Churn label | `is_churned` |
| Demographics | `gender`, `senior_citizen_flag`, `partner_flag`, `dependents_flag` |
| Subscription | `tenure_months`, `contract_type`, `internet_service_type`, `phone_service_flag`, `tech_support_flag` |
| Billing | `monthly_charges`, `total_charges`, `payment_method`, `paperless_billing_flag` |
| Activity RFM | `last_activity_date`, `days_since_last_activity`, `invoice_count`, `line_item_count`, `total_quantity`, `net_revenue`, `avg_order_value` |
| Product behavior | `distinct_product_count`, `return_count`, `return_rate`, `primary_country` |
| Support behavior | `ticket_count`, `open_ticket_count`, `critical_ticket_count`, `avg_first_response_minutes`, `avg_resolution_minutes`, `avg_satisfaction_rating` |
| Experience flags | `has_open_ticket_flag`, `has_critical_ticket_flag`, `low_satisfaction_flag`, `recent_support_ticket_flag` |
| Audit | `created_at`, `updated_at`, `data_quality_status` |

## 6. Architecture Diagram

```text
data/raw CSVs
     |
     v
staging models
stg_telco_customer | stg_customer_activity | stg_support_ticket
     |
     v
identity resolution
dim_customer + bridge_customer_identity
     |
     v
dimensions
dim_date | dim_product | dim_contract | dim_service_plan | dim_support_classification
     |
     v
facts
fact_customer_subscription_snapshot | fact_customer_activity_line | fact_support_ticket
     |
     v
churn_features_master
     |
     v
BI dashboards, business insights, and future ML modeling
```

## 7. Best Practices

- Preserve raw data without modification.
- Clean and type-cast in staging layers before modeling facts and dimensions.
- Define grain before keys and metrics.
- Use surrogate keys for analytical joins and retain source natural keys for lineage.
- Avoid direct cross-source joins until identity resolution is governed and tested.
- Generate features from facts using a snapshot date to prevent leakage.
- Add data quality checks for uniqueness, null keys, duplicate facts, invalid dates, and orphan foreign keys.